# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library. We will demonstrate how to load metadata, enumerate record sets and fields by their `@id`, extract data, perform exploratory data analysis (EDA), and create visualizations—all referencing dataset entities using their `@id` fields, as recommended for FAIR Croissant datasets.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.
We start by importing the required libraries and loading the metadata from the provided Croissant schema URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata
print(f"Dataset Name: {meta.name}\nDescription: {meta.description}")
print(f"Published: {meta.datePublished}\nIdentifier: {meta.identifier}\nKeywords: {meta.keywords}\nAuthors: {[a['@id'] for a in meta.author]}")

## 2. Data Overview
Review available record sets, fields, and their IDs.
Here, we enumerate record sets and their field `@id`s for inspection, referencing each by `@id` only (as recommended in FAIR Croissant best practice).

In [ ]:
# List available record sets and their fields
record_sets = meta.recordSet
if not record_sets:
    # If recordSet is empty, try loading from metadata via dataset.discovery
    print("No record sets found in the top-level metadata. Attempting to discover available record sets...")
    # Use the lower-level api to inspect what hasPart provides (common pattern in Croissant)
    if hasattr(meta, 'hasPart') and meta.hasPart:
        # Filter those which are record sets
        record_sets = [part for part in meta.hasPart if getattr(part, '@type', None) == 'RecordSet' or getattr(part, '@type', None) == 'cr:RecordSet']
else:
    record_sets = meta.recordSet

if not record_sets:
    # Attempt fallback: direct underlying discovery (this will work if mlcroissant is up-to-date)
    record_sets = [rs for rs in dataset._dataset_json['hasPart'] if rs.get('@type') == 'cr:RecordSet']

print(f"Number of record sets found: {len(record_sets)}")

record_set_ids = []
record_set_field_ids = {}
for rs in record_sets:
    if isinstance(rs, dict):
        record_set_id = rs.get('@id')
    else:
        record_set_id = getattr(rs, '@id', None)
    if not record_set_id:
        continue
    record_set_ids.append(record_set_id)
    # Try to get the fields
    # mlcroissant API should provide .field or .fields (list of field objects with @id)
    field_ids = []
    # Try to access rs.field (sometimes rs['field'])
    fields = None
    if hasattr(rs, 'field'):
        fields = rs.field
    elif isinstance(rs, dict):
        fields = rs.get('field', [])
    if fields:
        for f in fields:
            if isinstance(f, dict):
                fid = f.get('@id', str(f))
            else:
                fid = getattr(f, '@id', str(f))
            field_ids.append(fid)
    record_set_field_ids[record_set_id] = field_ids

print("Record set @ids and their field @ids:")
for rid in record_set_ids:
    print(f"  RecordSet: {rid}")
    print(f"    Fields: {record_set_field_ids[rid]}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We use only the `@id` notation for all entities.

**Note:** Some datasets contain a single record set. For this dataset, we select the `@id` of the main record set.

In [ ]:
# Usually, the main record set `@id` is clear — extract dataframes for all record sets

# If you know the specific @id, set it here. Otherwise, select the first from above.
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Selected main record set: {main_record_set_id}")
else:
    raise Exception("No record sets discovered in this dataset.")

# Load all record sets into a dictionary of DataFrames
dataframes = {}
for rsid in record_set_ids:
    print(f"Loading record set {rsid} ...")
    records = list(dataset.records(record_set=rsid))
    # Some record sets may be empty
    if records:
        dataframes[rsid] = pd.DataFrame(records)
    else:
        print(f"No records found for record set {rsid}.")

# Preview columns available in main record set dataframe
if main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    print(f"Columns in main record set DataFrame: {df.columns.tolist()}")
    display(df.head())
else:
    print(f"No dataframe for record set: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Use only `@id` field references throughout.

Let's proceed with EDA. We'll identify a numeric field by its `@id`, filter by a threshold, normalize, and group by another field if available.

In [ ]:
# --- Identify a numeric field (@id) and a group field (@id) ---
df = dataframes[main_record_set_id]

# For demonstration, attempt to auto-detect a numeric field, or specify one by @id
numeric_field_id = None
for col in df.columns:
    # Only try the column if it's numeric
    try:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    except Exception:
        continue
if not numeric_field_id:
    print("No numeric field auto-detected. Please set 'numeric_field_id' to the desired @id of a numeric field.")
else:
    print(f"Numeric field selected for EDA: {numeric_field_id}")

threshold = df[numeric_field_id].mean() if numeric_field_id else 10
print(f"Filtering records in {numeric_field_id} > {threshold}")
filtered_df = df[df[numeric_field_id] > threshold] if numeric_field_id else df
print(f"Filtered records: {filtered_df.shape[0]}")

if numeric_field_id:
    # Normalize
    normalized_field = f"{numeric_field_id}_normalized"
    filtered_df[normalized_field] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, normalized_field]].head())

    # Attempt grouping by a categorical/text field (@id)
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and (df[col].dtype == object or pd.api.types.is_categorical_dtype(df[col])):
            group_field_id = col
            break
    if group_field_id:
        print(f"Grouping by field: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(grouped_df.head())
    else:
        print("No suitable group field detected for grouping.")

## 5. Visualization
Visualize a numeric field's distribution or explore relationships between fields (using only @id references).

Below is an example histogram for the selected numeric field. Adjust field IDs as appropriate for different analysis tasks.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

This notebook demonstrated how to access, explore, and analyze the FAIR² dataset using `mlcroissant`, referencing all dataset entities by their `@id` as per FAIR and Croissant standards. Users can extend this workflow further for machine learning, statistical analysis, or downstream FAIR data integration.